# Registro de errores – `01_log_error`

## Objetivo

El notebook `01_log_error` gestiona los errores generados durante la ejecución del pipeline y registra la información correspondiente en la tabla `workspace.metadata.etl_log`.

Su ejecución se realiza cuando una tarea predecesora del Workflow presenta un error, permitiendo conservar la trazabilidad de la ejecución y el detalle de la excepción.

## Flujo de procesamiento

1. **Obtención de parámetros de ejecución**

   * Recupera el `log_id` generado por `00_prepare_pipeline_control`.
   * Obtiene del Workflow el `error_code`, `run_id` y `result_state`.

2. **Configuración de la tabla de registro**

   * Define `workspace.metadata.etl_log` como tabla destino para registrar el resultado de la ejecución.

3. **Consulta del detalle del error**

   * Utiliza la Databricks Jobs API (`runs/get-output`) para consultar el output del Task que presentó el error.
   * La consulta utiliza el `run_id` correspondiente al Task fallido.

4. **Extracción del detalle de la excepción**

   * Obtiene el mensaje de error desde la respuesta de la API.
   * Si no se encuentra información detallada, registra un mensaje indicando que no fue posible obtener el detalle mediante la API.

5. **Actualización del registro de ejecución**

   * Actualiza el registro identificado mediante `log_id`.
   * Cambia el estado a `ERROR`.
   * Registra la fecha y hora de finalización.
   * Almacena en `mensaje_error` la información técnica de la falla, incluyendo:

     * Task que presentó el error.
     * Código del error.
     * `run_id`.
     * Resultado de la ejecución.
     * Detalle de la excepción.

6. **Validación**

   * Consulta el registro actualizado para verificar que el error haya sido almacenado correctamente.
   * Se dispone adicionalmente de una consulta SQL para revisar el historial completo de ejecuciones registradas.

## Resultado

El proceso permite mantener un registro centralizado de los errores del pipeline, facilitando la trazabilidad, diagnóstico y auditoría de las ejecuciones fallidas.

El flujo general es:

`Task fallido → 01_log_error → Databricks Jobs API → detalle de excepción → ETL_LOG`


In [0]:
# ============================================================
# OBTENCIÓN DE PARÁMETROS DE LA EJECUCIÓN
# ============================================================

# 1. Recupera el log_id generado por la tarea que falló
try:
    log_id = dbutils.jobs.taskValues.get(
        taskKey="ingest_sharepoint_bronze",
        key="log_id",
        debugValue=""
    )
except Exception:
    log_id = None

# 2. Inicializa y lee los widgets enviados por el Workflow
dbutils.widgets.text("error_code", "UNKNOWN_ERROR")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("result_state", "FAILED")

error_code = dbutils.widgets.get("error_code")
result_state = dbutils.widgets.get("result_state")

# 3. Prioriza recuperar el run_id numérico publicado en taskValues
try:
    run_id = dbutils.jobs.taskValues.get(
        taskKey="ingest_sharepoint_bronze",
        key="run_id",
        debugValue=""
    )
except Exception:
    run_id = None

# Si no vino por taskValues, lee del widget enviado por el orquestador
if not run_id or run_id == "" or run_id == "MANUAL_RUN":
    run_id = dbutils.widgets.get("run_id")

print(f"✓ log_id: {log_id}")
print(f"✓ error_code: {error_code}")
print(f"✓ run_id: {run_id}")
print(f"✓ result_state: {result_state}")

In [0]:
# ============================================================
# CONFIGURACIÓN Y CONSULTA DEL DETALLE DEL TASK FALLIDO VÍA API
# ============================================================

import requests

TABLA_ETL_LOG = "prod_dataco.metadata.etl_log"
error_detail = None

# Valida que exista un run_id numérico válido antes de hacer la petición HTTP
if run_id and str(run_id).isdigit() and str(run_id) != "0":
    try:
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        token = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .apiToken()
            .get()
        )

        url = f"https://{workspace_url}/api/2.2/jobs/runs/get-output"

        response = requests.get(
            url,
            headers={"Authorization": f"Bearer {token}"},
            params={"run_id": str(run_id)}
        )

        if response.status_code == 200:
            output_info = response.json()
            error_detail = (
                output_info.get("error")
                or output_info.get("error_trace")
                or (output_info.get("notebook_output") or {}).get("result")
            )
        else:
            error_detail = f"Error al consultar la API (Status {response.status_code}): {response.text}"

    except Exception as ex:
        error_detail = f"Excepción durante la conexión a Jobs API: {str(ex)}"

# Mensaje de respaldo si no se obtuvo una traza específica
if not error_detail:
    error_detail = "No fue posible obtener la traza del error mediante Jobs API (Run ID no válido o sin respuesta)."

print("✓ Detalle del error extraído:")
print(error_detail)

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL REGISTRO DE ERROR EN METADATA
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

TASK_ORIGEN = "01_ingest_sharepoint_bronze"

if log_id:
    delta_log = DeltaTable.forName(spark, TABLA_ETL_LOG)

    mensaje_error = (
        f"Task: {TASK_ORIGEN} | "
        f"Error code: {error_code} | "
        f"Run ID: {run_id} | "
        f"Resultado: {result_state} | "
        f"Detalle: {error_detail}"
    )

    delta_log.update(
        condition=F.col("log_id") == log_id,
        set={
            "fecha_fin": F.current_timestamp(),
            "estado": F.lit("ERROR"),
            "mensaje_error": F.lit(mensaje_error[:2000]) # Trunca a 2000 caracteres para evitar desbordamiento
        }
    )

    print("✓ Registro de ejecución actualizado")
    print(f"✓ log_id: {log_id}")
    print("✓ Estado: ERROR")
else:
    print("⚠️ No se realizó la actualización: log_id no fue recibido.")

In [0]:
# ============================================================
# VALIDACIÓN DEL LOG DE ERROR
# ============================================================

from pyspark.sql import functions as F

if log_id:
    display(
        spark.table(TABLA_ETL_LOG)
        .filter(F.col("log_id") == log_id)
    )

In [0]:
# ============================================================
# ALERTA A SLACK
# ============================================================

import requests as slack_requests

try:
    webhook_url = dbutils.secrets.get(scope="alerts", key="slack_webhook_url")

    slack_payload = {
        "text": (
            f":rotating_light: *Fallo en pipeline DataCo*\n"
            f"*Task:* prepare_pipeline_control\n"
            f"*Estado:* {result_state}\n"
            f"*Error code:* {error_code}\n"
            f"*Run ID:* {run_id}\n"
            f"*log_id:* {log_id}"
        )
    }

    slack_response = slack_requests.post(webhook_url, json=slack_payload, timeout=10)

    if slack_response.status_code == 200:
        print("✓ Alerta enviada a Slack")
    else:
        print(f"⚠️ Slack respondió con status {slack_response.status_code}")

except Exception as ex:
    print(f"⚠️ No se pudo enviar la alerta a Slack (scope/clave no configurado aún): {ex}")